In [18]:
import pandas as pd 
import numpy as np
import pickle as pk 
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, brier_score_loss
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

pd.options.display.max_columns = 100
pd.options.display.max_rows = 60


# Preface: Evaluating Fairness Metrics on an eICU Prediction Task
In this notebook, you will:
1. Train a simple prediction model on real-world ICU data
2. Evaluate the model using several common algorithmic fairness metrics
3. Understand the equations behind these metrics
4. Reflect on when each fairness notion may or may not make sense in healthcare


In machine learning, people often talk about building a "fair" model. We will see in this notebook that fairness has been traditionally viewed as a single metric.

In this notebook, we will build intution for the fact that:
- a model can look fair under one metric and unfair under another
- fairness depends on the clinical context and is not straightforward / one-size-fits-all


---

# Load the data

In this notebook, our task will be to predict Y= `mortality_48h`. 
That is, using data from the first 24 hours of hospitalization, predict whether the patient will die within 48 hours.
However, at the end you will change the target variable and reconsider fairness for this context. 


In [21]:
patient_agg = pd.read_parquet('../../X/clean_dataset.parquet')
feature_sets = pk.load(open('../../data/feature_names.pkl','rb'))
patient_agg

,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_ARDS-adult respiratory distress syndrome, non-cardiogenic pulmonary edema",admissiondx_Abdomen only trauma,admissiondx_Abdomen/extremity trauma,admissiondx_Abdomen/face trauma,admissiondx_Abdomen/multiple trauma,admissiondx_Abdomen/pelvis trauma,admissiondx_Abdomen/spinal trauma,admissiondx_Ablation or mapping of cardiac conduction pathway,"admissiondx_Abscess, neurologic","admissiondx_Abscess/infection-cranial, surgery for",admissiondx_Acid-base/electrolyte disturbance,admissiondx_Addisons disease,admissiondx_Adrenal neoplasm (including pheochromocytoma),admissiondx_Adrenalectomy,admissiondx_Alcohol withdrawal,admissiondx_Amputation (non-traumatic),admissiondx_Amyotrophic lateral sclerosis,admissiondx_Anaphylaxis,"admissiondx_Anastomosis, vascular",admissiondx_Anemia,"admissiondx_Aneurysm repair, ventricular","admissiondx_Aneurysm, abdominal aortic","admissiondx_Aneurysm, abdominal aortic; with dissection","admissiondx_Aneurysm, abdominal aortic; with rupture","admissiondx_Aneurysm, dissecting aortic","admissiondx_Aneurysm, thoracic aortic","admissiondx_Aneurysm, thoracic aortic; with dissection","admissiondx_Aneurysm, thoracic aortic; with rupture","admissiondx_Aneurysm/pseudoaneurysm, other","admissiondx_Aneurysms, repair of other (except ventricular)","admissiondx_Angina, stable (asymp or stable pattern of symptoms w/meds)","admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic and Mitral valve replacement,admissiondx_Aortic valve replacement (isolated),"admissiondx_Apnea, sleep","admissiondx_Apnea-sleep; surgery for (i.e., UPPP - uvulopalatopharyngoplasty)",admissiondx_Appendectomy,"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Arteriovenous malformation, surgery for","admissiondx_Arthritis, rheumatoid","admissiondx_Arthritis, septic",...,calcium,cd 4,chloride,cortisol,creatinine,direct bilirubin,eos,ethanol,fibrinogen,folate,free T4,glucose,glucose CSF,haptoglobin,ionized calcium,lactate,lipase,lymphs,magnesium,monos,myoglobin,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,prealbumin,prolactin,protein CSF,protein C,protein S,reticulocyte count,salicylate,serum ketones,serum osmolality,sodium,total bilirubin,total cholesterol,total protein,transferrin,triglycerides,troponin I,troponin T,uric acid,urinary creatinine,urinary osmolality,urinary sodium,urinary specific gravity
0,128919,Female,70,Caucasian,59,2015,1,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,101.500000,NaN,2.125000,NaN,0.500000,NaN,NaN,NaN,NaN,113.0,NaN,NaN,NaN,NaN,NaN,12.500000,NaN,16.500000,NaN,NaN,NaN,NaN,NaN,211.000000,70.5,4.100000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,139.000000,3.35,NaN,7.10,NaN,NaN,NaN,NaN,8.1,NaN,NaN,NaN,NaN
1,128927,Female,52,Caucasian,60,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,8.100000,NaN,107.750000,NaN,0.700000,NaN,3.000000,234.0,NaN,NaN,NaN,74.5,NaN,NaN,NaN,NaN,NaN,45.000000,1.900,7.000000,NaN,NaN,NaN,NaN,NaN,273.000000,45.0,3.750000,NaN,NaN,NaN,NaN,NaN,NaN,2.3,NaN,NaN,144.500000,0.40,NaN,7.40,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,128941,Male,68,Caucasian,73,2015,0,>= 500,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.200000,NaN,103.500000,NaN,2.725000,0.1,0.500000,NaN,NaN,NaN,NaN,151.5,NaN,NaN,NaN,1.666667,NaN,3.500000,1.200,4.500000,NaN,NaN,NaN,NaN,NaN,265.500000,91.5,4.300000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.500000,0.40,NaN,7.45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.018
3,128943,Male,71,Caucasian,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,9.000000,NaN,98.000000,NaN,0.865000,NaN,0.000000,NaN,NaN,NaN,NaN,153.0,NaN,NaN,NaN,0.800000,NaN,3.00

In [23]:
# This distinction of numerical (continuous), binary, and categorical columns may be useful for you 
num_cols = feature_sets['labs'].tolist() + feature_sets['vitals'].tolist() + ['age', 'admissionheight', 'admissionweight']
bin_cols = feature_sets['icd10_before24h'].tolist() + feature_sets['admissiondx'].tolist()
cat_cols = ['hospital_region', 'ethnicity', 'gender', 'hospital_numbedscategory', 'hospitaldischargeyear', 'hospitalid']

# For privacy I think, the dataset sets the age of everyone over 89 as the string '>89'. We set back to a continuous number 
if 'age' in patient_agg.columns: 
    patient_agg.loc[patient_agg.age == '> 89', 'age'] = 90
    patient_agg['age'] = patient_agg['age'].astype(float)


In [24]:
Y_label = 'mortality_at48h'
X_covariate_names = num_cols + bin_cols + cat_cols

In [25]:
X = patient_agg[X_covariate_names].copy()
Y = patient_agg[Y_label].copy()
X

,24 h urine protein,24 h urine urea nitrogen,ALT (SGPT),ANF/ANA,AST (SGOT),Acetaminophen,Amikacin peak,Amikacin random,Amikacin trough,BNP,BUN,Base Deficit,Base Excess,CPK,CPKMB,CPKMB INDEX,CRP,CRPhs,Carbamazepine,Carboxyhemoglobin,Clostridium difficile toxin A+B,Cyclosporin,Device,Digoxin,ESR,Fe,Fe/TIBC Ratio,Ferritin,FiO2,Gentamicin peak,Gentamicin random,Gentamicin trough,HCO3,HDL,HIV 1&2 AB,HSV 1&2 IgG AB,HSV 1&2 IgG AB titer,Hct,Hgb,LDH,LDL,LPM O2,Legionella pneumophila Ab,Lidocaine,Lithium,MCH,MCHC,MCV,MPV,Methemoglobin,...,admissiondx_Splenectomy,admissiondx_Stereotactic procedure,admissiondx_Subarachnoid hemorrhage/arteriovenous malformation,admissiondx_Subarachnoid hemorrhage/intracranial aneurysm,"admissiondx_Subarachnoid hemorrhage/intracranial aneurysm, surgery for","admissiondx_TURP, transurethral prostate resection for benign prostatic hypertrophy","admissiondx_TURP, transurethral prostate resection for cancer","admissiondx_Tamponade, pericardial","admissiondx_Thoracotomy for benign tumor (i.e. mediastinal chest wall mass, thymectomy)",admissiondx_Thoracotomy for bronchopleural fistula,admissiondx_Thoracotomy for esophageal cancer,admissiondx_Thoracotomy for lung cancer,admissiondx_Thoracotomy for lung reduction,admissiondx_Thoracotomy for other malignancy in chest,admissiondx_Thoracotomy for other reasons,admissiondx_Thoracotomy for pleural disease,admissiondx_Thoracotomy for thoracic/respiratory infection,admissiondx_Thrombectomy (with general anesthesia),admissiondx_Thrombectomy (without general anesthesia),admissiondx_Thrombocytopenia,"admissiondx_Thrombosis, vascular (deep vein)","admissiondx_Thrombus, arterial",admissiondx_Thyroid neoplasm,admissiondx_Thyroidectomy,admissiondx_Thyroidectomy and Parathyroidectomy,"admissiondx_Toxicity, drug (i.e., beta blockers, calcium channel blockers, etc.)",admissiondx_Tracheostomy,admissiondx_Transphenoidal surgery,"admissiondx_Transplant, other","admissiondx_Trauma medical, other","admissiondx_Trauma surgery, other",admissiondx_Tricuspid valve surgery,"admissiondx_Tumor removal, intracardiac","admissiondx_Ulcer disease, peptic","admissiondx_Vascular medical, other","admissiondx_Vascular surgery, other",admissiondx_Vasculitis,admissiondx_Vena cava clipping,admissiondx_Vena cava filter insertion,admissiondx_Ventricular Septal Defect (VSD) Repair,admissiondx_Ventriculostomy,admissiondx_Weaning from mechanical ventilation (transfer from other unit or hospital only),admissiondx_Whipple-surgery for pancreatic cancer,admissiondx_nan,hospital_region,ethnicity,gender,hospital_numbedscategory,hospitaldischargeyear,hospitalid
0,NaN,NaN,199.0,NaN,468.5,NaN,NaN,NaN,NaN,NaN,26.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.250000,13.150,NaN,NaN,NaN,NaN,NaN,NaN,29.20,32.650000,89.350000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Midwest,Caucasian,Female,<100,2015,59
1,NaN,NaN,52.0,NaN,40.0,NaN,NaN,NaN,NaN,NaN,15.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,43.600000,15.500,NaN,NaN,NaN,NaN,NaN,NaN,33.70,35.600000,94.800000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,Midwest,Caucasian,Female,<100,2015,60
2,NaN,NaN,19.5,NaN,19.5,NaN,NaN,NaN,NaN,NaN,36.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.300000,9.350,NaN,NaN,NaN,NaN,NaN,NaN,26.40,33.050000,80.050000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,Midwest,Caucasian,Male,>= 500,2015,73
3,NaN,NaN,18.5,NaN,16.0,NaN,NaN,NaN,NaN,24.0,15.000000,NaN,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,NaN,NaN,NaN,NaN,34.050000,10.900,NaN,NaN,3.0,NaN,NaN,NaN,27.65,32.050000,86.250000,NaN,NaN,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,

# Q1: Data preprocessing 
Right now dim(X) is huge with 1386 columns, highly sparse, and contains many missing features. 
We will need to do some preprocessing before we can train a model.  

In [ ]:
na_drop_threshold = 0.95
bin_thresh = 0.01

mask = X.isna().mean() > na_drop_threshold
print(f"Dropping columns whose missing rate >{na_drop_threshold}: ", X.columns[mask])
X = X.drop(columns=X.columns[mask])
mask = ~X[bin_cols].mean().between(bin_thresh, 1 - bin_thresh)
print(f"Dropping binary columns whose mean before imputation are not in [{bin_thresh}, 1-{bin_thresh}]: ", X[bin_cols].columns[mask])
X = X.drop(columns=X[bin_cols].columns[mask])
mask = ~X[bin_cols].mean().between(bin_thresh, 1 - bin_thresh)
print(f"Dropping binary columns whose mean before imputation are not in [{bin_thresh}, 1-{bin_thresh}]: ", X[cat_cols].columns[mask])
X = X.drop(columns=X[cat_cols].columns[mask])

num_cols = np.intersect1d(X.columns, num_cols).tolist()
cat_cols = np.intersect1d(X.columns, cat_cols).tolist()
bin_cols = np.intersect1d(X.columns, bin_cols).tolist()

X

In [ ]:
# TODO: Choose your own way to preprocess X such that it no longer contains missing values and has reduced dimensionalitiy.
# You are free to use the above way but you can also do something else. 
# The goal is to get a dataset that is more manageable (for NxM matrix, a large M can lead to issues in modeling)
# This can be either by hand-selecting what features are relevant for Y, 
# removing features with low information (ie. high missingness rates, low variance, etc.)

# Q2: Train a naive logistic regression model to predict mortality


In [ ]:
### TODO: Use sklearn to build a simple logistic regression model to predict Y given X. 
# You can use your solution from week1 notebook. 
# Please use train_test_split to separate into training and test sets. 
## Output the AUROC and AUPRC score on the test set. 
## Hint: I recommend using StandardScaler for normalizing continuous features 

# Introducing traditional fairness metrics 
In this section, we go through several common fairness metrics and if they make sense in a healthcare setting.

All of these metrics depend on:
- Y: the true outcome (e.g., mortality)
- $\hat{Y}$: the model prediction (0 or 1)
- A: a group attribute (e.g., race, gender)

We will compare these metrics across groups (e.g., If A is gender, we can compare across two gender categories observed in the data). 

------------------------------------------------------------
Demographic parity
------------------------------------------------------------

Definition: Satisfying demographic parity means

$$P(\hat{Y} = 1 | A = a) = P(\hat{Y} = 1 | A = b)$$



Empirical estimate of demographic parity in a dataset:

$$DP(a) = DP(b) \, \forall a,b \in A$$
where 
$$DP(a) = \frac{1 }{n_a} * \sum_{i : A_i = a} 1(\hat{Y}_i = 1)$$


Note: This is equivalent to the criteria $\hat{Y} \perp A$. 


In words:
A model with demographic parity would predict Y is "positive" (e.g., high risk) at the same rate across groups.


Interpretation:
This metric ignores the true outcome Y.
It only checks whether the model assigns positive predictions equally across groups.

Why this may be problematic in healthcare:
If groups have different true risk levels (e.g., due to structural inequities),
forcing equal prediction rates may lead to:
- under-treatment of higher-risk groups
- over-treatment of lower-risk groups


In [29]:
# If A = gender, these are all possible A=a values 
patient_agg.gender.value_counts(dropna=False)

gender
Male       72733
Female     62961
None          68
Unknown       10
Other          6
Name: count, dtype: int64

In [30]:
# If A = ethnicity, these are all possible A=a values 
patient_agg.ethnicity.value_counts(dropna=False)

ethnicity
Caucasian           104452
African American     15132
Other/Unknown         6296
Hispanic              5051
Asian                 2286
None                  1531
Native American       1030
Name: count, dtype: int64

# Q3: Estimate demographic parity
Note there are two parts, the calculation and the reflection. 

In [ ]:
# TODO: Using the above equation, calculate the score DP(a) for all A, for both race and for sex. 

In [ ]:
# TODO:  Reflection:
# Should a mortality prediction model flag high risk at the same rate across all groups,
# even if underlying risk differs?


------------------------------------------------------------
True positive rate parity (Equal Opportunity)
------------------------------------------------------------

Definition:

$$P(\hat{Y}= 1 | Y = 1, A = a) = P(\hat{Y}= 1 | Y = 1, A = b)$$



Empirical estimate in a dataset:

$$TPR(a) = TPR(b) \, \forall a,b \in A$$
where 
$$TPR(a) = \frac{1 }{n_a} * \sum_{i : A_i = a, Y_i=1} 1(\hat{Y}_i = 1)$$


Note: This is equivalent to the criteria $\hat{Y} \perp A \mid Y=1$. 



In words:
Among patients who truly experience the outcome (e.g., pass away in the hospital),
the model should identify them at equal rates across groups.

Interpretation:
This focuses on correctly identifying *true cases*.

Why this matters in healthcare:
Missing a high-risk patient (false negative) can be dangerous.
Equal opportunity ensures that one group is not systematically under-detected.



# Q4: Estimate true positive rate parity (equal opportunity)
Note there are two parts, the calculation and the reflection. 

In [ ]:
# TODO: Using the above equation, calculate the score TPR(a) for all A, for both race and for sex. 

In [ ]:
# TODO:  Reflection:
# For a mortality prediction model, is it more important to equalize:
# - overall prediction rates?
# - or the ability to detect true high-risk patients?


------------------------------------------------------------
False positive rate parity 
------------------------------------------------------------

Definition:

$$P(\hat{Y}= 1 | Y = 0, A = a) = P(\hat{Y}= 0 | Y = 0, A = b)$$



Empirical estimate in a dataset:

$$FPR(a) = FPR(b) \, \forall a,b \in A$$
where 
$$FPR(a) = \frac{1 }{n_a} * \sum_{i : A_i = a, Y_i=0} 1(\hat{Y}_i = 1)$$


Note: This is equivalent to the criteria $\hat{Y} \perp A \mid Y=0$. 




In words:
Among patients who do NOT experience the outcome,
how often does the model incorrectly predict high risk?

Interpretation:
This measures over-prediction.

Why this matters in healthcare:
False positives can lead to:
- unnecessary interventions
- alarm fatigue
- misallocation of limited resources






# Q5: Estimate false positive rate parity 
Note there are two parts, the calculation and the reflection. 

In [ ]:
# TODO: Using the above equation, calculate the score FPR(a) for all A, for both race and for sex. 

In [ ]:
# TODO:  Reflection:
# When (for our case of mortality, or for other cases of Y) might it be important to ensure that one group is not over-treated
# relative to another?

------------------------------------------------------------
Equalized odds
------------------------------------------------------------

Definition: Equalized odds would enforce 

$$P(\hat{Y}= 1 | Y = y, A = a) = P(\hat{Y} = 1 | Y = y, A = b)  \, \, y \in {0,1}$$

Empirical estimate in a dataset:

$$EO(a,y) = EO(b,y) \, \forall a,b \in A \, \, for y \in {0,1}$$
where 
$$EO(a,y) = \frac{1 }{n_{a,y}} * \sum_{i : A_i = a, Y_i=y} 1(\hat{Y}_i = 1)$$

Note: This is equivalent to the criteria $\hat{Y} \perp A \mid Y$. 

This means:
- equal TPR across groups
- equal FPR across groups

Interpretation:
The model should have similar error behavior across groups.

Why this is appealing:
It balances both:
- detecting true cases (TPR)
- avoiding false alarms (FPR)

Why this is difficult:
Equalized odds is stricter than just equal TPR or equal FPR. Because the constraints are stronger, enforcing it can conflict with:
- calibration
- overall accuracy

# Q6: Estimate equalized odds
Note there are two parts, the calculation and the reflection. 

In [ ]:
# TODO: Using the above equation, calculate the score FPR(a) for all A, for both race and for sex. 

In [ ]:
# TODO: Reflection:
# Would equalized odds be a good goal for mortality prediction?
# What tradeoffs might arise?

------------------------------------------------------------
Calibration
------------------------------------------------------------

Definition: 

$$P(Y = 1 | R = r, A = a) = r$$

where R is the predicted risk score (i.e. the continuous output in [0,1] of a logistic regression model, 
                                     prior to thresholding into 0 or 1). 


In words:
If the model predicts 0.2 risk, then about 20% of those patients
should actually experience the outcome — and this should hold across groups.

Interpretation:
Calibration is about whether predicted probabilities are meaningful.

Why this matters in healthcare:
Many clinical models output probabilities (risk scores),
which are used directly in decision-making.

Important note:
Calibration may conflict with equalized odds when base rates differ across groups.


# Q7: Tradeoffs

Enforcing fairness in a model often involves tradeoffs. In fact, there is a well-known [“impossibility theorem”](https://arxiv.org/abs/1609.07236) showing that certain fairness criteria (e.g., calibration and equalized odds) cannot all be satisfied at the same time when groups have different base rates. Similarly, there is often a tradeoff between training a “fair” model and training a highly accurate model. For example, enforcing certain fairness constraints may require the model to ignore predictive signals that differ across groups, which can reduce overall performance.



In [ ]:
# TODO:
# Did you observe in your evaluation that some metrics did better (showing the model was "more fair" for that metric) than other metrics? 

# Q7: Who should we prioritize for fairness? 
So far, we (along with much of the algorithmic fairness literature) have evaluated fairness with respect to attributes such as gender and race/ethnicity, in part because these variables are what are available in the data.



In [ ]:
# TODO:
# Do you think that gender and race/ethnicity are the most important groups to consider in a healthcare setting?
# - Are there other patient characteristics or subgroups that should be prioritized when evaluating fairness?
# - What factors (e.g., clinical, social, or structural) might make a group particularly important to protect?


In [126]:
data_apache = data[['mortality_at48h', 'patienthealthsystemstayid']].merge(
    apache_ypred, on='patienthealthsystemstayid', how='inner')
y_ = data_apache.mortality_at48h
y_proba = data_apache.predictedicumortality
print(f"APACHE AUROC:", roc_auc_score(y_, y_proba))
print(f"APACHE AUPRC:", average_precision_score(y_, y_proba))

APACHE AUROC: 0.8613640402333097
APACHE AUPRC: 0.3256122955269497


In [81]:
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_validate, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LassoCV


preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median", add_indicator=False), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse=True), cat_cols),
        ("bin", SimpleImputer(strategy="constant", fill_value=0, add_indicator=False), bin_cols),
    ]
)
data_pp = preprocess.fit_transform(data[num_cols + cat_cols + bin_cols])
data_pp = pd.DataFrame(data_pp, columns = preprocess.get_feature_names_out())
data_pp

,num__ALT (SGPT),num__AST (SGOT),num__BNP,num__BUN,num__Base Deficit,num__Base Excess,num__CPK,num__CPKMB,num__CPKMB INDEX,num__Carboxyhemoglobin,num__FiO2,num__HCO3,num__HDL,num__Hct,num__Hgb,num__LDL,num__LPM O2,num__MCH,num__MCHC,num__MCV,num__MPV,num__Methemoglobin,num__O2 Content,num__O2 Sat (%),num__PEEP,num__PT,num__PT INR,num__PTT,num__RBC,num__RDW,num__Respiratory Rate,num__TSH,num__TV,num__Temperature,num__Total CO2,num__Vent Rate,num__WBC x 1000,num__WBC's in urine,num__admissionheight,num__admissionweight,num__age,num__albumin,num__alkaline phos.,num__ammonia,num__anion gap,num__bands,num__basos,num__bedside glucose,num__bicarbonate,num__calcium,...,bin__K76.9,bin__K92.2,bin__N17.9,bin__N18.6,bin__N18.9,bin__N30.9,bin__N39.0,bin__R00.0,bin__R07.9,bin__R10.9,bin__R11.0,bin__R41.82,bin__R50.9,bin__R56.9,bin__R65.2,bin__R65.20,bin__R65.21,bin__R73.9,bin__S06.5,bin__S06.9,"bin__admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",bin__admissiondx_Aortic valve replacement (isolated),"bin__admissiondx_Arrest, respiratory (without cardiac arrest)","bin__admissiondx_Bleeding, GI-location unknown","bin__admissiondx_Bleeding, lower GI","bin__admissiondx_Bleeding, upper GI","bin__admissiondx_CABG alone, coronary artery bypass grafting","bin__admissiondx_CHF, congestive heart failure","bin__admissiondx_CVA, cerebrovascular accident/stroke",bin__admissiondx_Cardiac arrest (with or without respiratory arrest; for respiratory arrest see Respiratory System),"bin__admissiondx_Coma/change in level of consciousness (for hepatic see GI, for diabetic see Endocrine, if related to cardiac arrest, see CV)",bin__admissiondx_Diabetic ketoacidosis,"bin__admissiondx_Embolus, pulmonary",bin__admissiondx_Emphysema/bronchitis,bin__admissiondx_Head only trauma,"bin__admissiondx_Hemorrhage/hematoma, intracranial","bin__admissiondx_Hypertension, uncontrolled (for cerebrovascular accident-see Neurological System)","bin__admissiondx_Infarction, acute myocardial (MI)","bin__admissiondx_Pneumonia, bacterial","bin__admissiondx_Renal failure, acute","bin__admissiondx_Respiratory - medical, other","bin__admissiondx_Rhythm disturbance (atrial, supraventricular)",bin__admissiondx_Rhythm disturbance (conduction defect),bin__admissiondx_Seizures (primary-no structural brain disease),"bin__admissiondx_Sepsis, GI","bin__admissiondx_Sepsis, cutaneous/soft tissue","bin__admissiondx_Sepsis, pulmonary","bin__admissiondx_Sepsis, renal/UTI (including bladder)","bin__admissiondx_Sepsis, unknown",bin__admissiondx_nan
0,199.0,468.5,505.525,26.500000,3.909091,-0.70,127.416667,4.20,2.9,1.05,40.000,23.026136,40.0,40.250000,13.150,83.0,4.0,29.20,32.650000,89.350000,9.80,0.45,15.9,97.000,5.000,21.233333,2.033333,29.000000,4.5050,17.950000,18.0,1.600,500.0000,37.0,25.0,16.0,12.250000,4.0,152.4,84.3,70.0,3.20,149.5,36.0,17.50,7.0,0.0,136.750000,24.000000,9.000000,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,52.0,40.0,505.525,15.750000,3.909091,-0.70,127.416667,4.20,2.9,1.05,40.000,23.026136,40.0,43.600000,15.500,83.0,4.0,33.70,35.600000,94.800000,9.80,0.45,15.9,97.000,5.000,13.950000,1.140000,30.800000,4.6000,11.900000,18.0,1.600,500.0000,37.0,25.0,16.0,7.600000,4.0,162.6,54.4,52.0,4.00,81.0,36.0,17.25,7.0,0.0,136.750000,23.500000,8.100000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,19.5,19.5,505.525,36.000000,3.909091,-0.70,127.416667,4.20,2.9,1.05,40.000,23.026136,40.0,28.300000,9.350,83.0,4.0,26.40,33.050000,80.050000,9.80,0.45,15.9,97.000,5.000,12.100000,1.100000,24.000000,3.5350,15.850000,18.0,1.600,500.0000,37.0,20.0,16.0,12.950000,4.0,180.3,73.9,68.0,2.65,63.0,36.0,16.00,7.0,0.0,141.000000,19.500000,9.200000,.

In [82]:
bin_cols_ = [x for x in data_pp.columns if 'bin_' in x]
mask = data_pp[bin_cols_].mean().between(bin_thresh, 1 - bin_thresh)
print(f"Dropping binary columns whose mean before imputation are not in [{bin_thresh}, 1-{bin_thresh}]: ", data_pp[bin_cols_].columns[mask])
# data_pp = data_pp.drop(columns=data_pp[bin_cols].columns[mask])

Dropping binary columns whose mean before imputation are not in [0.01, 1-0.01]:  Index(['bin__A41.9', 'bin__D69.6', 'bin__D72.829', 'bin__E03.9', 'bin__E10.1',
       'bin__E46', 'bin__E78.5', 'bin__E83.42', 'bin__E86.0', 'bin__E86.1',
       'bin__E87.0', 'bin__E87.1', 'bin__E87.2', 'bin__E87.5', 'bin__E87.6',
       'bin__F10.239', 'bin__F32.9', 'bin__G93.40', 'bin__I10', 'bin__I21.3',
       'bin__I21.4', 'bin__I25.10', 'bin__I26.99', 'bin__I46.9', 'bin__I48.0',
       'bin__I50.1', 'bin__I50.9', 'bin__I62.9', 'bin__I63.50', 'bin__I67.8',
       'bin__I95.9', 'bin__J18.9', 'bin__J44.1', 'bin__J44.9', 'bin__J45',
       'bin__J69.0', 'bin__J91.8', 'bin__J96.00', 'bin__J96.91', 'bin__K92.2',
       'bin__N17.9', 'bin__N18.6', 'bin__N18.9', 'bin__N30.9', 'bin__N39.0',
       'bin__R00.0', 'bin__R07.9', 'bin__R10.9', 'bin__R41.82', 'bin__R50.9',
       'bin__R56.9', 'bin__R65.2', 'bin__R65.20', 'bin__R65.21', 'bin__R73.9',
       'bin__S06.5', 'bin__S06.9',
       'bin__admissiondx_Angi

In [87]:
model.coef_.shape

(1, 445)

In [128]:

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.preprocessing import StandardScaler

X = data_pp
y = data['mortality_at48h']
print(X.shape, y.shape)
model = LogisticRegressionCV(random_state=44, max_iter=100, n_jobs=-1, penalty='l1',
                             cv=3,
                             solver='saga')


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_.squeeze(), 
                        'abs_coef': np.abs(model.coef_).squeeze()}).sort_values('abs_coef')
from sklearn.metrics import roc_auc_score, average_precision_score
for X_, name, y_ in [[X_train, 'train', y_train], [X_test, 'test', y_test]]:
    y_proba = model.predict_proba(X_)[:, 1]
    print(f"{name.upper()} AUROC:", roc_auc_score(y_, y_proba))
    print(f"{name.upper()} AUPRC:", average_precision_score(y_, y_proba))

(135778, 445) (135778,)


/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_mode

TRAIN AUROC: 0.8654173337315287
TRAIN AUPRC: 0.31944541643316593
TEST AUROC: 0.859360393222773
TEST AUPRC: 0.30186139274328927


/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [136]:

X = data_pp.drop(columns=[c for c in X.columns if 'gender' in c])
y = data['mortality_at48h']
print(X.shape, y.shape)
model = LogisticRegressionCV(random_state=44)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_.squeeze(), 
                        'abs_coef': np.abs(model.coef_).squeeze()}).sort_values('abs_coef')
from sklearn.metrics import roc_auc_score, average_precision_score
for X_, name, y_ in [[X_train, 'train', y_train], [X_test, 'test', y_test]]:
    y_proba = model.predict_proba(X_)[:, 1]
    print(f"{name.upper()} AUROC:", roc_auc_score(y_, y_proba))
    print(f"{name.upper()} AUPRC:", average_precision_score(y_, y_proba))
    
     
mask = ~data.gender.isin(['Female', 'Male'])
X = scaler.transform(X[mask])
y = y[mask]
y_proba = model.predict_proba(X)[:, 1]   
display(data[mask])
print(f"AUROC:", roc_auc_score(y, y_proba))
print(f"AUPRC:", average_precision_score(y, y_proba))

print("------"*10)


X = data_pp.drop(columns=[c for c in data_pp.columns if 'gender' in c])[data.gender.isin(['Female', 'Male'])]
Xheldout = data_pp.drop(columns=[c for c in data_pp.columns if 'gender' in c])[~data.gender.isin(['Female', 'Male'])]
y = data['mortality_at48h'][data.gender.isin(['Female', 'Male'])]
yheldout = data['mortality_at48h'][~data.gender.isin(['Female', 'Male'])]
print(X.shape, y.shape)
model = LogisticRegressionCV(random_state=44)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)
Xheldout= scaler.transform(Xheldout)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_.squeeze(), 
                        'abs_coef': np.abs(model.coef_).squeeze()}).sort_values('abs_coef')
from sklearn.metrics import roc_auc_score, average_precision_score
for X_, name, y_ in [[X_train, 'train', y_train], [X_test, 'test', y_test], [Xheldout, 'heldout', yheldout]]:
    y_proba = model.predict_proba(X_)[:, 1]
    print(f"{name.upper()} AUROC:", roc_auc_score(y_, y_proba))
    print(f"{name.upper()} AUPRC:", average_precision_score(y_, y_proba))

(135778, 440) (135778,)


/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regr

TRAIN AUROC: 0.875091396266053
TRAIN AUPRC: 0.34138334502729983
TEST AUROC: 0.8637864148143812
TEST AUPRC: 0.3095319675260235


,patienthealthsystemstayid,gender,age,ethnicity,hospitalid,hospitaldischargeyear,mortality_at48h,hospital_numbedscategory,hospital_region,"admissiondx_Angina, unstable (angina interferes w/quality of life or meds are tolerated poorly)",admissiondx_Aortic valve replacement (isolated),"admissiondx_Arrest, respiratory (without cardiac arrest)","admissiondx_Bleeding, GI-location unknown","admissiondx_Bleeding, lower GI","admissiondx_Bleeding, upper GI","admissiondx_CABG alone, coronary artery bypass grafting","admissiondx_CHF, congestive heart failure","admissiondx_CVA, cerebrovascular accident/stroke",admissiondx_Cardiac arrest (with or without respiratory arrest; for respiratory arrest see Respiratory System),"admissiondx_Coma/change in level of consciousness (for hepatic see GI, for diabetic see Endocrine, if related to cardiac arrest, see CV)",admissiondx_Diabetic ketoacidosis,"admissiondx_Embolus, pulmonary",admissiondx_Emphysema/bronchitis,admissiondx_Head only trauma,"admissiondx_Hemorrhage/hematoma, intracranial","admissiondx_Hypertension, uncontrolled (for cerebrovascular accident-see Neurological System)","admissiondx_Infarction, acute myocardial (MI)","admissiondx_Pneumonia, bacterial","admissiondx_Renal failure, acute","admissiondx_Respiratory - medical, other","admissiondx_Rhythm disturbance (atrial, supraventricular)",admissiondx_Rhythm disturbance (conduction defect),admissiondx_Seizures (primary-no structural brain disease),"admissiondx_Sepsis, GI","admissiondx_Sepsis, cutaneous/soft tissue","admissiondx_Sepsis, pulmonary","admissiondx_Sepsis, renal/UTI (including bladder)","admissiondx_Sepsis, unknown",admissiondx_nan,admissionheight,admissionweight,A41.9,D62,D69.6,D72.829,E03.9,E10.1,E46,E78.5,E83.42,...,PEEP,PT,PT INR,PTT,RBC,RDW,Respiratory Rate,TSH,TV,Temperature,Total CO2,Vent Rate,WBC x 1000,WBC's in urine,albumin,alkaline phos.,ammonia,anion gap,bands,basos,bedside glucose,bicarbonate,calcium,chloride,creatinine,direct bilirubin,eos,fibrinogen,glucose,ionized calcium,lactate,lipase,lymphs,magnesium,monos,pH,paCO2,paO2,phosphate,platelets x 1000,polys,potassium,sodium,total bilirubin,total cholesterol,total protein,triglycerides,troponin I,troponin T,urinary specific gravity
781,136001,None,NaN,None,67,2015,0,None,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1911,146319,None,NaN,None,60,2014,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3461,159767,None,NaN,None,63,2014,0,100 - 249,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6843,190197,None,NaN,None,68,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7169,192868,None,NaN,None,68,2015,0,<100,Midwest,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

IndexError: boolean index did not match indexed array along dimension 0; dimension is 27156 but corresponding boolean dimension is 135778

------------------------------------------------------------
(135694, 440) (135694,)


/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regr

TRAIN AUROC: 0.8745169334905343
TRAIN AUPRC: 0.3333495562029647
TEST AUROC: 0.8680627648054795
TEST AUPRC: 0.3312141938411203
HELDOUT AUROC: 0.9958847736625515
HELDOUT AUPRC: 0.9166666666666665


In [135]:
coef_df[coef_df.coef==0].feature.tolist()

['cat__hospitalid_207',
 'cat__hospitalid_258',
 'cat__hospitalid_256',
 'cat__hospitalid_254',
 'cat__hospitalid_253',
 'bin__F32.9',
 'cat__hospitalid_251',
 'cat__hospitalid_250',
 'cat__hospitalid_249',
 'cat__hospitalid_262',
 'cat__hospitalid_248',
 'cat__hospitalid_245',
 'cat__hospitalid_244',
 'bin__F43.0',
 'cat__hospitalid_227',
 'cat__hospitalid_226',
 'cat__hospitalid_224',
 'cat__hospitalid_220',
 'cat__hospitalid_215',
 'cat__hospitalid_246',
 'cat__hospitalid_212',
 'cat__hospitalid_263',
 'cat__hospitalid_265',
 'cat__hospitalid_307',
 'cat__hospitalid_303',
 'cat__hospitalid_301',
 'cat__hospitalid_300',
 'cat__hospitalid_283',
 'cat__hospitalid_282',
 'cat__hospitalid_281',
 'cat__hospitalid_280',
 'cat__hospitalid_264',
 'cat__hospitalid_279',
 'cat__hospitalid_275',
 'bin__E87.6',
 'cat__hospitalid_272',
 'cat__hospitalid_271',
 'cat__hospitalid_269',
 'cat__hospitalid_268',
 'cat__hospitalid_267',
 'cat__hospitalid_266',
 'cat__hospitalid_277',
 'cat__hospitalid_2

In [131]:
coef_df[-50:]

,feature,coef,abs_coef
80,num__phosphate,0.024596,0.024596
5,num__Base Excess,-0.025309,0.025309
108,num__systemicsystolic_bin3,-0.025479,0.025479
257,cat__hospitalid_273,0.026641,0.026641
396,bin__K92.2,-0.028695,0.028695
43,num__ammonia,0.029012,0.029012
26,num__PT INR,0.031645,0.031645
134,cat__hospital_numbedscategory_>= 500,0.034861,0.034861
176,cat__hospitalid_131,0.035093,0.035093
23,num__O2 Sat (%),-0.039024,0.039024


In [ ]:
model = LogisticRegressionCV(random_state=44)


X_train, X_test, y_train, y_test = train_test_split(
    X.to_numpy(), y, test_size=0.2, stratify=y, random_state=42
)
model.fit(X_train, y_train)

coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_, 'abs_coef': np.abs(model.coef_)}).sort_values('abs_coef')
from sklearn.metrics import roc_auc_score, average_precision_score
for X_, name, y_ in [[X_train, 'train', y_train], [X_test, 'test', y_test]]:
    y_proba = model.predict_proba(X_)[:, 1]
    print(f"{name.upper()} AUROC:", roc_auc_score(y_, y_proba))
    print(f"{name.upper()}  AUPRC:", average_precision_score(y_, y_proba))

In [91]:

from sklearn.linear_model import LogisticRegression, LogisticRegressionCV


X = data_pp.drop(columns=[c for c in data_pp.columns if 'hospitalid' in c])
y = data['mortality_at48h']
print(X.shape, y.shape)
model = LogisticRegressionCV(random_state=44, penalty='l1', solver='saga', max_iter=1000)


for id, counts in data.hospitalid.value_counts().items():
    if counts > 1000:
        print(f"id: {id}")
        mask = data.hospitalid==id 
        X_ = X[mask].to_numpy()
        y_ = y[mask]
        model.fit(X_, y[mask])
        y_proba = model.predict_proba(X_)[:, 1]
        print(f"{id} AUROC:", roc_auc_score(y_, y_proba))
        print(f"{id} AUPRC:", average_precision_score(y_, y_proba))
        # coef_df = pd.DataFrame({'feature': X.columns, 'coef': model.coef_, 'abs_coef': np.abs(model.coef_)}).sort_values('abs_coef')
        # from sklearn.metrics import roc_auc_score, average_precision_score
        for id2, counts2 in data.hospitalid.value_counts().items():
            if id2 != id and counts2 > 1000:
                X_ = X[data.hospitalid==id2]
                y_ = y[data.hospitalid==id2]
                y_proba = model.predict_proba(X_)[:, 1]
                print(f"--{id2} AUROC:", roc_auc_score(y_, y_proba))
                print(f"--{id2} AUPRC:", average_precision_score(y_, y_proba))

(135778, 237) (135778,)
id: 73


/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


73 AUROC: 0.7492905242905243
73 AUPRC: 0.06069736663812419
--264 AUROC: 0.6549538953868036
--264 AUPRC: 0.06718386275164905
--420 AUROC: 0.6400001324222682
--420 AUPRC: 0.1571124271854858
--338 AUROC: 0.688504459164773
--338 AUPRC: 0.09697750758784682
--243 AUROC: 0.5932073299433944
--243 AUPRC: 0.03376402037024185
--458 AUROC: 0.5914739932606012
--458 AUPRC: 0.1564388195947931
--167 AUROC: 0.6509894654213262
--167 AUPRC: 0.11586667151812187
--300 AUROC: 0.6158037854444232
--300 AUPRC: 0.10547826445311492
--443 AUROC: 0.6591313235608538
--443 AUPRC: 0.12146705437718476
--188 AUROC: 0.6233207134878335
--188 AUPRC: 0.10390622897400481
--208 AUROC: 0.5177187909209952
--208 AUPRC: 0.07463489248449368
--252 AUROC: 0.6577388033069647
--252 AUPRC: 0.1417512748573732
--199 AUROC: 0.6864808751638671
--199 AUPRC: 0.1499358005861629
--122 AUROC: 0.5719243480722235
--122 AUPRC: 0.10509696215159262
--176 AUROC: 0.6440007829320807
--176 AUPRC: 0.09746879508851283
--281 AUROC: 0.6426723682821244
--28

/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feat

264 AUROC: 0.5279311559146219
264 AUPRC: 0.05564369168612817
--73 AUROC: 0.5642496392496392
--73 AUPRC: 0.01571240047201466
--420 AUROC: 0.5624463689814079
--420 AUPRC: 0.130778322425189
--338 AUROC: 0.4684972626789764
--338 AUPRC: 0.03963649694370807
--243 AUROC: 0.3960891777799098
--243 AUPRC: 0.018340354050234643
--458 AUROC: 0.506630041111074
--458 AUPRC: 0.09053485537214476
--167 AUROC: 0.5802948771913858
--167 AUPRC: 0.11077791906630508
--300 AUROC: 0.5764730976868093
--300 AUPRC: 0.12631966314997467
--443 AUROC: 0.4810808132955784
--443 AUPRC: 0.06041949255341303
--188 AUROC: 0.5124626636480503
--188 AUPRC: 0.07046905030922897
--208 AUROC: 0.49737650953004514
--208 AUPRC: 0.041096617343884936
--252 AUROC: 0.47842938995618467
--252 AUPRC: 0.07864354113893628
--199 AUROC: 0.5300805290246753
--199 AUPRC: 0.09272919318353044
--122 AUROC: 0.5375030398711493
--122 AUPRC: 0.07867530851700813
--176 AUROC: 0.40861225288706204
--176 AUPRC: 0.03201838066932628
--281 AUROC: 0.49997360363214

...

122 AUROC: 0.5701163722476708
122 AUPRC: 0.1090565167707222
--73 AUROC: 0.6417408209074876
--73 AUPRC: 0.04126063624861136
--264 AUROC: 0.623520019693183
--264 AUPRC: 0.06168678787180708
--420 AUROC: 0.5985479898299697
--420 AUPRC: 0.1504438332574134
--338 AUROC: 0.5875604637627961
--338 AUPRC: 0.0724859378168459
--243 AUROC: 0.5917742012856184
--243 AUPRC: 0.03763732668361838
--458 AUROC: 0.5773603650693031
--458 AUPRC: 0.15469267666236325
--167 AUROC: 0.6341651403283238
--167 AUPRC: 0.10018359880197031
--300 AUROC: 0.6147480104472619
--300 AUPRC: 0.08682890369932476
--443 AUROC: 0.5920054963679126
--443 AUPRC: 0.09046050477485915
--188 AUROC: 0.5971519735608348
--188 AUPRC: 0.11682351168341168
--208 AUROC: 0.4747972137349047
--208 AUPRC: 0.06438634510791597
--252 AUROC: 0.6438061321596384
--252 AUPRC: 0.13179349550072358
--199 AUROC: 0.6183589385925825
--199 AUPRC: 0.12588731939654946
--176 AUROC: 0.5741162654139753
--176 AUPRC: 0.07768346255654188
--281 AUROC: 0.5845519656495266
--2

/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feat

176 AUROC: 0.5497044431395577
176 AUPRC: 0.06676709921202491
--73 AUROC: 0.6033189033189034
--73 AUPRC: 0.04692744954316813
--264 AUROC: 0.5231497468263546
--264 AUPRC: 0.06508982711302322
--420 AUROC: 0.5887533767678372
--420 AUPRC: 0.14454274111731363
--338 AUROC: 0.5056110854860405
--338 AUPRC: 0.048853822435882904
--243 AUROC: 0.3897450350187086
--243 AUPRC: 0.017321094685263244
--458 AUROC: 0.5429636673005706
--458 AUPRC: 0.12343586111821524
--167 AUROC: 0.5871961069839934
--167 AUPRC: 0.1072385308132234
--300 AUROC: 0.5773109656351376
--300 AUPRC: 0.13494373478991717
--443 AUROC: 0.4809713031189541
--443 AUPRC: 0.06698156294091798
--188 AUROC: 0.5418452515226708
--188 AUPRC: 0.07739178081455751
--208 AUROC: 0.5039443292594209
--208 AUPRC: 0.06017060195171911
--252 AUROC: 0.46729341382660244
--252 AUPRC: 0.07426814786249739
--199 AUROC: 0.5244719117086748
--199 AUPRC: 0.08822900269485358
--122 AUROC: 0.5721157104527713
--122 AUPRC: 0.11065399720479033
--281 AUROC: 0.51411325801569

/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/base.py:457: UserWarning: X has feature names, but LogisticRegressionCV was fitted without feature names
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/kara/anaconda3/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:350: 

KeyboardInterrupt: 

In [ ]:
for p in os.listdir('./eicu-collaborative-research-database-demo-2.0.1/'):
    if not p.endswith('.csv.gz'):
        print(p)
        continue
    data_dir = './eicu-collaborative-research-database-demo-2.0.1/'
    df = pd.read_csv(f'{data_dir}/{p}')
    print(p)
    display(df.head())

diagnosis.csv.gz


,diagnosisid,patientunitstayid,activeupondischarge,diagnosisoffset,diagnosisstring,icd9code,diagnosispriority
0,7607199,346380,False,5028,cardiovascular|ventricular disorders|hypertension,"401.9, I10",Other
1,7570429,346380,False,685,neurologic|altered mental status / pain|change...,"780.09, R41.82",Major
2,7705483,346380,True,5035,cardiovascular|shock / hypotension|hypotension,"458.9, I95.9",Major
3,7848601,346380,True,5035,neurologic|altered mental status / pain|schizo...,"295.90, F20.9",Major
4,7451475,346380,False,5028,pulmonary|disorders of vasculature|pulmonary e...,"415.19, I26.99",Major


vitalAperiodic.csv.gz


,vitalaperiodicid,patientunitstayid,observationoffset,noninvasivesystolic,noninvasivediastolic,noninvasivemean,paop,cardiacoutput,cardiacinput,svr,svri,pvr,pvri
0,3661418,141764,81,171.0,90.0,116.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3661424,141764,334,153.0,78.0,103.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3661417,141764,77,176.0,87.0,107.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3661419,141764,165,173.0,106.0,128.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3661421,141764,255,182.0,103.0,133.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


admissionDx.csv.gz


...

medication.csv.gz


/var/folders/gv/cvbvg4412fg914zd3_cc80lh0000gn/T/ipykernel_22833/313154529.py:6: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f'{data_dir}/{p}')


,medicationid,patientunitstayid,drugorderoffset,drugstartoffset,drugivadmixture,drugordercancelled,drugname,drughiclseqno,dosage,routeadmin,frequency,loadingdose,prn,drugstopoffset,gtc
0,7278819,141765,134,1396,No,No,WARFARIN SODIUM 5 MG PO TABS,2812.0,5 3,PO,NaN,NaN,No,2739,0
1,9726266,141765,1,-188,No,No,5 ML VIAL : DILTIAZEM HCL 25 MG/5ML IV SOLN,182.0,15 3,IV,Once PRN,NaN,Yes,171,38
2,10293599,141765,115,856,No,No,ASPIRIN EC 81 MG PO TBEC,1820.0,81 3,PO,Daily,NaN,No,2739,0
3,10871534,141765,114,316,No,No,DILTIAZEM HCL 30 MG PO TABS,182.0,30 3,PO,Q6H SCH,NaN,No,2739,0
4,10128716,141765,115,856,No,No,LISINOPRIL 5 MG PO TABS,132.0,5 3,PO,Daily,NaN,No,2428,0


intakeOutput.csv.gz


,intakeoutputid,patientunitstayid,intakeoutputoffset,intaketotal,outputtotal,dialysistotal,nettotal,intakeoutputentryoffset,cellpath,celllabel,cellvaluenumeric,cellvaluetext
0,9314532,147307,-394,0.0,0.0,0.0,0.0,-394,flowsheet|Flowsheet Cell Labels|I&O|Weight|Bod...,Bodyweight (lb),159.8,159.8
1,9314533,147307,-394,0.0,0.0,0.0,0.0,-394,flowsheet|Flowsheet Cell Labels|I&O|Weight|Bod...,Bodyweight (kg),72.5,72.5
2,9319306,211715,1533,120.0,0.0,0.0,120.0,1533,flowsheet|Flowsheet Cell Labels|I&O|Intake (ml...,P.O.,120.0,120.0
3,9319891,219981,6504,120.0,0.0,0.0,120.0,6504,flowsheet|Flowsheet Cell Labels|I&O|Intake (ml...,P.O.,120.0,120.0
4,9319987,158057,624,0.0,0.0,0.0,0.0,624,flowsheet|Flowsheet Cell Labels|I&O|Weight|Bod...,Bodyweight (lb),359.0,359.0


### apache table

In [ ]:
df = pd.read_csv(f'{data_dir}/apachePatientResult.csv.gz').drop(columns='apachepatientresultsid')
df = df[df.patientunitstayid.isin(patient.patientunitstayid.unique())]
df = patient[['patientunitstayid', 'patienthealthsystemstayid', 'mortality_at48h']].merge(df, on=['patientunitstayid'], how='left')
df = df[df.apacheversion=='IVa']
apache_ypred = df.groupby('patienthealthsystemstayid')[['predictedicumortality']].mean().reset_index()
display(df)

,patientunitstayid,patienthealthsystemstayid,mortality_at48h,apachepatientresultsid,physicianspeciality,physicianinterventioncategory,acutephysiologyscore,apachescore,apacheversion,predictedicumortality,actualicumortality,predictediculos,actualiculos,predictedhospitalmortality,actualhospitalmortality,predictedhospitallos,actualhospitallos,preopmi,preopcardiaccath,ptcawithin24h,unabridgedunitlos,unabridgedhosplos,actualventdays,predventdays,unabridgedactualventdays
1,141168,128919,1,26571.0,critical care medicine (CCM),Unknown,49.0,65.0,IVa,0.028889,EXPIRED,3.091127,2.4972,0.059099,EXPIRED,6.628720,2.4972,0.0,0.0,0.0,2.4972,2.4972,NaN,NaN,NaN
5,141194,128941,0,53136.0,critical care medicine (CCM),Unknown,57.0,70.0,IVa,0.046448,ALIVE,4.167129,3.3423,0.102283,ALIVE,12.978228,9.2167,0.0,0.0,0.0,3.3423,9.2167,NaN,NaN,NaN
8,141203,128948,0,8.0,hospitalist,I,73.0,90.0,IVa,0.291609,ALIVE,8.670299,1.2979,0.470973,ALIVE,16.319389,3.7493,0.0,0.0,0.0,1.2979,3.7493,2.0,5.738093,2.0
10,141227,128968,0,138142.0,internal medicine,Unknown,83.0,100.0,IVa,0.335934,ALIVE,8.209624,1.1472,0.488562,ALIVE,16.508280,1.8861,0.0,0.0,0.0,1.1472,1.8861,2.0,4.474474,2.0
13,141233,128973,0,47819.0,surgery-cardiac,I,49.0,66.0,IVa,0.027697,ALIVE,3.107540,10.8923,0.035490,ALIVE,6.739768,15.9313,0.0,0.0,0.0,10.8923,15.9313,2.0,1.422043,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
263924,3353235,2743084,0,3267489.0,cardiology,II,30.0,35.0,IVa,0.011009,ALIVE,1.934817,0.7423,0.022630,ALIVE,5.759009,2.6701,0.0,0.0,0.0,0.7423,2.6701,NaN,NaN,NaN
263926,3353237,2743086,0,3246235.0,pulmonary/CCM,II,37.0,54.0,IVa,0.026984,ALIVE,2.566297,0.8812,0.062568,ALIVE,7.565244,6.7215,0.0,0.0,0.0,0.8812,6.7215,NaN,NaN,NaN
263928,3353251,2743099,0,3272802.0,cardiology,II,142.0,158.0,IVa,0.876797,ALIVE,8.920949,11.2909,0.918169,ALIVE,11.222566,13.4056,0.0,0.0,1.0,11.2909,13.4056,9.0,6.464630,9.0
263930,3353254,2743102,0,3262174.0,hospitalist,II,18.0,35.0,IVa,0.011004,ALIVE,2.045046,0.2993,0.025801,ALIVE,6.199350,4.4549,0.0,0.0,0.0,0.2993,4.4549,NaN,NaN,NaN


In [ ]:
lab = pd.read_csv(f'{data_dir}/apachePredVar.csv.gz')
lab = lab[lab.patientunitstayid.isin(patient.patientunitstayid.unique())]
lab = lab.merge(patient[['patientunitstayid', 'patienthealthsystemstayid']], on=['patientunitstayid'], how='left')
display(lab)
lab.patienthealthsystemstayid.nunique()


,apachepredvarid,patientunitstayid,sicuday,saps3day1,saps3today,saps3yesterday,gender,teachtype,region,bedcount,admitsource,graftcount,meds,verbal,motor,eyes,age,admitdiagnosis,thrombolytics,diedinhospital,aids,hepaticfailure,lymphoma,metastaticcancer,leukemia,immunosuppression,cirrhosis,electivesurgery,activetx,readmit,ima,midur,ventday1,oobventday1,oobintubday1,diabetes,managementsystem,var03hspxlos,pao2,fio2,ejectfx,creatinine,dischargelocation,visitnumber,amilocation,day1meds,day1verbal,day1motor,day1eyes,day1pao2,day1fio2,patienthealthsystemstayid
0,2406430,141178,1,0,0,0,1,0,3,9,8,3,-1,-1,-1,-1,52.0,NaN,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,-1.00,4,1,-1,-1,-1,-1,-1,-1.0,-1.0,128927
1,27799,141197,1,0,0,0,0,0,3,30,8,3,0,5,6,4,71.0,SEPSISPULM,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,-1.00,4,1,-1,0,5,6,4,-1.0,-1.0,128943
2,1786953,141227,1,0,0,0,0,0,3,9,4,3,0,4,6,3,82.0,SEPSISPULM,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,1,1,0,0,1,0,65.0,21.0,-1,1.90,6,1,-1,0,4,6,3,65.0,21.0,128968
3,7948,141229,1,0,0,0,1,0,3,43,8,3,0,5,6,4,NaN,CHF,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,1,1,0,0,1,0,-1.0,-1.0,-1,-1.00,4,1,-1,0,5,6,4,-1.0,-1.0,128970
4,11916,141260,1,0,0,0,1,0,3,19,8,3,0,5,6,4,43.0,ASTHMA,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,1.04,4,1,-1,0,5,6,4,-1.0,-1.0,128995
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64268,1806800,3353184,1,0,0,0,1,0,3,23,8,3,0,5,6,4,83.0,CARDIOMYOP,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,0.63,8,1,-1,0,5,6,4,-1.0,-1.0,2743043
64269,825962,3353235,1,0,0,0,0,0,3,23,8,3,0,5,6,4,50.0,CHF,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,-1.00,8,1,-1,0,5,6,4,-1.0,-1.0,2743084
64270,996721,3353237,1,0,0,0,1,0,3,11,7,3,0,4,6,4,79.0,PULMEMBOL,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,1.07,4,1,-1,0,4,6,4,-1.0,-1.0,2743086
64271,2418339,3353254,1,0,0,0,0,0,3,14,8,3,0,5,6,4,81.0,LOWGIBLEED,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,0,0,0,0,1,0,-1.0,-1.0,-1,-1.00,4,1,-1,0,5,6,4,-1.0,-1.0,2743102


63855

In [ ]:
lab.replace(-1, np.nan)

,labid,patientunitstayid,labresultoffset,labtypeid,labname,labresult,labresulttext,labmeasurenamesystem,labmeasurenameinterface,labresultrevisedoffset,hospitaladmitoffset,time_from_admit
1,50363251,141168,1133.0,3,pt - inr,2.5,2.5,ratio,NaN,1208.0,0.0,1133.0
3,50363250,141168,1133.0,3,pt,26.6,26.6,sec,sec,1208.0,0.0,1133.0
5,56072056,141168,231.0,3,pt - inr,1.7,1.7,ratio,NaN,268.0,0.0,231.0
8,47753731,141168,516.0,1,bun,26.0,26,mg/dL,mg/dL,540.0,0.0,516.0
11,56072055,141168,231.0,3,pt,17.1,17.1,sec,sec,268.0,0.0,231.0
...,...,...,...,...,...,...,...,...,...,...,...,...
28417385,826336611,3353263,-37.0,3,ptt,25.0,25,sec,sec,54.0,-108.0,71.0
28417387,824772677,3353263,-7.0,3,mch,27.0,27,pg,pg,6.0,-108.0,101.0
28417389,824772676,3353263,-7.0,3,rdw,13.3,13.3,%,%,6.0,-108.0,101.0
28417391,824772675,3353263,-7.0,3,wbc x 1000,6.4,6.4,K/mcL,K/uL,6.0,-108.0,101.0


In [ ]:
lab = pd.read_csv(f'{data_dir}/apachePredVar.csv.gz').drop(columns='apachepredvarid').replace(-1, np.nan)
# lab = pd.read_csv(f'{data_dir}/apacheApsVar.csv.gz').drop(columns='apacheapsvarid').replace(-1, np.nan)
lab = lab[lab.patientunitstayid.isin(patient.patientunitstayid.unique())]
# lab["labname"] = lab["labname"].str.lower().replace("-",'').replace(' ', '')
# lab = lab.merge(patient[['patientunitstayid', 'hospitaladmitoffset']], on='patientunitstayid', how='left')
# lab['time_from_admit'] = - lab.hospitaladmitoffset + lab.labresultoffset
# lab = lab[lab.time_from_admit <= hours_for_data*60]
print(lab.columns)
lab

Index(['patientunitstayid', 'sicuday', 'saps3day1', 'saps3today',
       'saps3yesterday', 'gender', 'teachtype', 'region', 'bedcount',
       'admitsource', 'graftcount', 'meds', 'verbal', 'motor', 'eyes', 'age',
       'admitdiagnosis', 'thrombolytics', 'diedinhospital', 'aids',
       'hepaticfailure', 'lymphoma', 'metastaticcancer', 'leukemia',
       'immunosuppression', 'cirrhosis', 'electivesurgery', 'activetx',
       'readmit', 'ima', 'midur', 'ventday1', 'oobventday1', 'oobintubday1',
       'diabetes', 'managementsystem', 'var03hspxlos', 'pao2', 'fio2',
       'ejectfx', 'creatinine', 'dischargelocation', 'visitnumber',
       'amilocation', 'day1meds', 'day1verbal', 'day1motor', 'day1eyes',
       'day1pao2', 'day1fio2'],
      dtype='object')


,patientunitstayid,sicuday,saps3day1,saps3today,saps3yesterday,gender,teachtype,region,bedcount,admitsource,graftcount,meds,verbal,motor,eyes,age,admitdiagnosis,thrombolytics,diedinhospital,aids,hepaticfailure,lymphoma,metastaticcancer,leukemia,immunosuppression,cirrhosis,electivesurgery,activetx,readmit,ima,midur,ventday1,oobventday1,oobintubday1,diabetes,managementsystem,var03hspxlos,pao2,fio2,ejectfx,creatinine,dischargelocation,visitnumber,amilocation,day1meds,day1verbal,day1motor,day1eyes,day1pao2,day1fio2
1,141178,1,0,0,0,1.0,0,3,9,8.0,3,NaN,NaN,NaN,NaN,52.0,NaN,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,4.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,141197,1,0,0,0,0.0,0,3,30,8.0,3,0.0,5.0,6.0,4.0,71.0,SEPSISPULM,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,4.0,1,NaN,0.0,5.0,6.0,4.0,NaN,NaN
6,141227,1,0,0,0,0.0,0,3,9,4.0,3,0.0,4.0,6.0,3.0,82.0,SEPSISPULM,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,1,1,0,0,1,0,65.0,21.0,NaN,1.90,6.0,1,NaN,0.0,4.0,6.0,3.0,65.0,21.0
7,141229,1,0,0,0,1.0,0,3,43,8.0,3,0.0,5.0,6.0,4.0,NaN,CHF,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,1,1,0,0,1,0,NaN,NaN,NaN,NaN,4.0,1,NaN,0.0,5.0,6.0,4.0,NaN,NaN
10,141260,1,0,0,0,1.0,0,3,19,8.0,3,0.0,5.0,6.0,4.0,43.0,ASTHMA,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,1.04,4.0,1,NaN,0.0,5.0,6.0,4.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
171161,3353184,1,0,0,0,1.0,0,3,23,8.0,3,0.0,5.0,6.0,4.0,83.0,CARDIOMYOP,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,0.63,8.0,1,NaN,0.0,5.0,6.0,4.0,NaN,NaN
171172,3353235,1,0,0,0,0.0,0,3,23,8.0,3,0.0,5.0,6.0,4.0,50.0,CHF,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,8.0,1,NaN,0.0,5.0,6.0,4.0,NaN,NaN
171173,3353237,1,0,0,0,1.0,0,3,11,7.0,3,0.0,4.0,6.0,4.0,79.0,PULMEMBOL,0,0,0,0,0,0,0,0,0,NaN,0,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,1.07,4.0,1,NaN,0.0,4.0,6.0,4.0,NaN,NaN
171175,3353254,1,0,0,0,0.0,0,3,14,8.0,3,0.0,5.0,6.0,4.0,81.0,LOWGIBLEED,0,0,0,0,0,0,0,0,0,NaN,1,0,0,0,0,0,0,0,1,0,NaN,NaN,NaN,NaN,4.0,1,NaN,0.0,5.0,6.0,4.0,NaN,NaN


In [ ]:
patient.hospitalid.value_counts(dropna=False)

hospitalid
146    40
123    30
171    25
157    25
167    24
       ..
408    10
384    10
383    10
397    10
259    10
Name: count, Length: 186, dtype: int64

In [ ]:
patient.ethnicity.value_counts(dropna=False)

ethnicity
Caucasian           2010
African American     231
Hispanic             115
Other/Unknown         83
NaN                   39
Asian                 30
Native American       12
Name: count, dtype: int64

In [ ]:
# Create a database connection using settings from config file
config='../db/config.ini'

# connection info
conn_info = dict()
if os.path.isfile(config):
    config = ConfigObj(config)
    conn_info["sqluser"] = config['username']
    conn_info["sqlpass"] = config['password']
    conn_info["sqlhost"] = config['host']
    conn_info["sqlport"] = config['port']
    conn_info["dbname"] = config['dbname']
    conn_info["schema_name"] = config['schema_name']
else:
    conn_info["sqluser"] = 'postgres'
    conn_info["sqlpass"] = ''
    conn_info["sqlhost"] = 'localhost'
    conn_info["sqlport"] = 5432
    conn_info["dbname"] = 'eicu'
    conn_info["schema_name"] = 'public,eicu_crd'
    
# Connect to the eICU database
print('Database: {}'.format(conn_info['dbname']))
print('Username: {}'.format(conn_info["sqluser"]))
if conn_info["sqlpass"] == '':
    # try connecting without password, i.e. peer or OS authentication
    try:
        if (conn_info["sqlhost"] == 'localhost') & (conn_info["sqlport"]=='5432'):
            con = psycopg2.connect(dbname=conn_info["dbname"],
                                   user=conn_info["sqluser"])            
        else:
            con = psycopg2.connect(dbname=conn_info["dbname"],
                                   host=conn_info["sqlhost"],
                                   port=conn_info["sqlport"],
                                   user=conn_info["sqluser"])
    except:
        conn_info["sqlpass"] = getpass.getpass('Password: ')

        con = psycopg2.connect(dbname=conn_info["dbname"],
                               host=conn_info["sqlhost"],
                               port=conn_info["sqlport"],
                               user=conn_info["sqluser"],
                               password=conn_info["sqlpass"])
query_schema = 'set search_path to ' + conn_info['schema_name'] + ';'

Database: eicu
Username: postgres


NameError: name 'getpass' is not defined

In [ ]:
conda create --name cs197-rag python=3.12
